# 15 · Capstone — End-to-End Mini Project: Fine-Tune a Small Model for Lead Intent Classification

In plain English, this is the **grand finale** where everything you learned across the whole course comes together into one real, runnable project. Up to now each notebook taught a single skill in isolation — how data is prepared, how a model is formatted, what LoRA is, how to evaluate, how to run inference. Here we **chain all of those skills into a single working pipeline** that takes raw sales leads and teaches a small AI model to sort them into `hot`, `warm`, and `cold` buckets so a sales team knows who to call first.

Think of the earlier notebooks as learning to chop, to sauté, to make a sauce, and to plate a dish. This notebook is **cooking the whole meal from start to finish** and serving it. By the end you'll have a fine-tuned model saved to disk, an evaluation report telling you how good it is, and a `predict()` function you can call on brand-new leads.

Everything here uses a **small, free model** (`distilbert-base-uncased`) and runs comfortably on a **plain laptop CPU** — no GPU required. Let's build it.

## What you'll build

By the end of this notebook you will have produced, with your own hands:

- A **synthetic lead dataset** of 200 sales leads, each with realistic fields and a `hot`/`warm`/`cold` label.
- A clean **train / validation split** saved as `leads_train.jsonl` and `leads_val.jsonl`.
- A **fine-tuned text classifier** (DistilBERT with 3 output classes) trained on your data.
- An **evaluation report**: accuracy, precision, recall, F1, a confusion-matrix plot, a manual-review table, and a comparison against a dumb baseline.
- A reusable **`predict(lead_dict)`** function that classifies brand-new leads and shows the model's confidence.
- The **saved model + tokenizer** on disk, ready to be wrapped in an API (notebook 14) and shipped.

This is exactly the shape of a real fine-tuning project you might do at work. The dataset is toy-sized so it runs fast, but the *steps* are the real thing.

## Prerequisites & recap — which earlier notebooks each step uses

This capstone deliberately **reuses** what you already practiced. If a step feels fuzzy, the notebook in parentheses is where it was taught in detail:

| Project step | What it does | Taught in |
|---|---|---|
| Step 2–3 | Generate, inspect, split & save data | **09 · Data preparation** |
| Step 5 | Turn each lead into a text string / prompt template | **10 · Formatting data for fine-tuning** |
| Step 4 & 7 | The optional LoRA / PEFT path for LLMs | **11 · LoRA** (and **12 · QLoRA**) |
| Step 8 | Accuracy, precision/recall/F1, confusion matrix | **13 · Evaluating fine-tuned models** |
| Step 9–10 | Running predictions and saving/serving the model | **14 · Inference & deployment** |

You should also be comfortable with the Hugging Face basics from **08** (`AutoTokenizer`, `AutoModel...`, `Trainer`, `save_pretrained`). We'll move a little faster on those since you've seen them.

**What you need installed:** the libraries in the next cell. Everything is free and CPU-friendly.

## Setup — install, import, and detect your hardware

Run the cell below once. The `%pip install` line is **commented out** — uncomment it if you're on Google Colab or a fresh machine. After installing you may need to restart the kernel.

The cell also **detects your hardware**. It will pick a GPU (`cuda`), an Apple-Silicon GPU (`mps`), or fall back to `cpu`. **This whole notebook is designed to finish on CPU in a few minutes**, so don't worry if it says `cpu` — that's the expected, supported path.

In [ ]:
# Uncomment on Colab or a fresh environment, then restart the kernel:
# %pip install transformers datasets peft accelerate scikit-learn matplotlib torch

import os, json, random                       # standard library: files, JSON, randomness
import numpy as np                            # arrays + math
import pandas as pd                           # tables for inspecting data
import torch                                  # the engine models run on
import matplotlib.pyplot as plt               # plotting (confusion matrix)

import transformers, datasets                 # Hugging Face core libraries
print("transformers:", transformers.__version__)
print("datasets:    ", datasets.__version__)
print("torch:       ", torch.__version__)

# --- Device detection: pick the fastest thing available, else CPU ---
if torch.cuda.is_available():
    DEVICE = "cuda"                           # NVIDIA GPU
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    DEVICE = "mps"                            # Apple-Silicon GPU
else:
    DEVICE = "cpu"                            # everyone has this
print("Using device:", DEVICE)
print("Note: CPU is fully supported here — the dataset and model are tiny on purpose.")

## Step 1 — Define the problem

**The business situation.** Imagine you work at a company that sells, say, home solar systems. Every day your marketing brings in hundreds of *leads* — people who filled in a form, downloaded a brochure, or asked for a quote. Your sales team is small and can only call a fraction of them today. **Who should they call first?**

That's the whole job: look at what we know about each lead and predict how likely they are to buy soon. We'll bucket every lead into one of three **intent** classes:

- **`hot`** — call today, they're ready.
- **`warm`** — interested but needs nurturing.
- **`cold`** — low priority for now.

**The inputs (features) we have for each lead:**

| Field | Meaning | Example |
|---|---|---|
| `family_size` | people in the household | `4` |
| `income` | yearly household income | `85000` |
| `rent_or_own` | do they rent or own their home | `"own"` |
| `cta` | the last call-to-action they took | `"requested_quote"` |
| `engagement_month` | month they last engaged | `"May"` |
| `current_condition` | how they describe their situation | `"urgent"` |

**Why this is valuable.** A model that ranks leads well lets a small team spend their limited hours on the people most likely to convert — directly more sales for the same effort. This is a textbook **multi-class classification** problem (3 classes), which is exactly what a fine-tuned classifier is good at.

## Step 2 — Generate & inspect the data

Real projects start from a CSV or a database export (that was the focus of **notebook 09**). To keep this notebook self-contained and reproducible, we **generate** a realistic synthetic dataset instead.

The generator below builds each lead with random-but-sensible fields, then computes a hidden **score** from simple business rules (a quote request is worth more than a newsletter signup, an urgent buyer is hotter than a browser, etc.) and turns that score into the `hot`/`warm`/`cold` label. Because there's a real underlying pattern, the model has something genuine to learn — but the pattern is fuzzy enough that the model won't get a perfect score, which is realistic.

> We seed the randomness (`random.seed(42)`) so **you get the exact same 200 leads every time** you run this — reproducibility, just like in notebook 09.

In [ ]:
import random
random.seed(42)
MONTHS = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
CTAS = ["requested_quote","booked_demo","downloaded_brochure","newsletter_signup"]
CONDITIONS = ["urgent","exploring","just_browsing"]

def make_lead():
    family_size = random.randint(1, 6)
    income = random.choice([25000,40000,55000,70000,85000,100000,120000,150000])
    rent_or_own = random.choice(["rent","own"])
    cta = random.choice(CTAS)
    engagement_month = random.choice(MONTHS)
    current_condition = random.choice(CONDITIONS)
    score = 0
    if cta in ("requested_quote","booked_demo"): score += 2
    elif cta == "downloaded_brochure": score += 1
    if rent_or_own == "own": score += 1
    if income >= 80000: score += 1
    if family_size >= 4: score += 1
    if current_condition == "urgent": score += 2
    elif current_condition == "exploring": score += 1
    lead_intent = "hot" if score >= 5 else ("warm" if score >= 3 else "cold")
    return {"family_size": family_size, "income": income, "rent_or_own": rent_or_own,
            "cta": cta, "engagement_month": engagement_month,
            "current_condition": current_condition, "lead_intent": lead_intent}

leads = [make_lead() for _ in range(200)]
print("Generated", len(leads), "leads. First one:")
print(leads[0])

**What this does:** it builds a list of 200 dictionaries, each one lead. The `score` logic is the *hidden truth* the model will try to rediscover from the features alone (the model never sees the score — only the fields and the final label).

Now let's **inspect** the data the way you always should before training — with pandas. We look at a few rows and, crucially, the **class balance** (`value_counts`). If one class dominates, accuracy numbers become misleading, so we always check.

In [ ]:
df = pd.DataFrame(leads)
print("Shape (rows, columns):", df.shape)
display(df.head())                            # first 5 leads as a table

print("\nClass balance (how many of each label):")
print(df["lead_intent"].value_counts())
print("\nAs percentages:")
print((df["lead_intent"].value_counts(normalize=True) * 100).round(1))

**What this does:** `df.head()` shows the first rows so you can sanity-check the fields look right, and `value_counts()` tells us how many `hot` / `warm` / `cold` we have. You'll likely see the classes are **somewhat imbalanced** — that's normal for real lead data (there are always more lukewarm leads than red-hot ones). We'll keep this in mind when we evaluate.

> **Reality check:** 200 examples is *small*. With this little data the model can learn the gist but won't be razor-sharp, and the numbers will wobble from run to run. That's a genuine and important lesson: **more data almost always helps**. We use 200 so the notebook runs in minutes; a real project would use thousands.

## Step 3 — Save as JSONL and make a train / validation split

This is straight from **notebook 09**. We split the data into:

- a **training set** (~80%) the model *learns* from, and
- a **validation set** (~20%) the model *never trains on*, which we use to measure honest performance.

We save each split as **JSONL** (one JSON object per line) — the standard, tool-friendly format for fine-tuning datasets. We shuffle with a fixed seed first so the split is reproducible, then confirm the class balance survived in *both* splits (we don't want all the `hot` leads to land in only one side).

In [ ]:
# Shuffle reproducibly, then slice 80/20.
rng = random.Random(123)                      # separate seeded RNG just for the split
shuffled = leads[:]                           # copy so we don't disturb the original list
rng.shuffle(shuffled)

split_idx = int(0.8 * len(shuffled))          # 80% mark
train_leads = shuffled[:split_idx]            # first 80%
val_leads   = shuffled[split_idx:]            # last 20%
print(f"Train: {len(train_leads)} leads   |   Validation: {len(val_leads)} leads")

def write_jsonl(path, rows):
    """Write a list of dicts to a JSONL file (one compact JSON object per line)."""
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")

write_jsonl("leads_train.jsonl", train_leads)
write_jsonl("leads_val.jsonl",   val_leads)
print("Saved leads_train.jsonl and leads_val.jsonl")

In [ ]:
# Confirm the class balance held up in BOTH splits.
print("Train balance:")
print(pd.Series([r["lead_intent"] for r in train_leads]).value_counts())
print("\nValidation balance:")
print(pd.Series([r["lead_intent"] for r in val_leads]).value_counts())

**What this does:** writes two files to disk and prints the label counts in each. As long as every class appears in both splits, we're good. (For tiny or very imbalanced datasets you'd use a *stratified* split — scikit-learn's `train_test_split(..., stratify=labels)` — to guarantee proportions match. With 200 rows and 3 classes our simple shuffle is fine.)

### ✏️ Exercise — peek at the raw files
Open `leads_train.jsonl` in a text editor (or run `!head -n 3 leads_train.jsonl` in a new cell). Confirm each line is one self-contained JSON object. This is the exact format you'd hand to almost any fine-tuning tool.

## Step 4 — Choose the approach (and why)

There are **two reasonable ways** to fine-tune for this task. Knowing *when* to pick which is a real skill.

**Path A — A small encoder classifier (DistilBERT) — ✅ our primary path.**
DistilBERT is a tiny, fast "reading" model. We bolt a 3-way classification head on top and fine-tune it to read a lead's description and output one of `cold/warm/hot`. It's purpose-built for exactly this — *map text to a label* — and it trains in **minutes on CPU**.

**Path B — Instruction-tune a generative LLM with LoRA (notebooks 11–12) — optional extension.**
Here you'd phrase the task as a prompt ("Given this lead, answer hot/warm/cold:") and fine-tune a small *generative* LLM (e.g. with LoRA so it's cheap). This is powerful and flexible, but it's **overkill** for a clean 3-class problem on structured fields: it's slower, heavier, and the model can output text that isn't even one of your three labels (you'd have to parse/clean it).

**Our decision:** for a small, well-defined **classification** task on structured data, the **encoder classifier (Path A) wins** — simpler, faster, and the output is guaranteed to be one of your 3 classes. We'll build Path A end-to-end and sketch Path B as a stretch goal so you see how they relate.

> Rule of thumb: *Need a fixed set of categories?* Reach for a classifier first. *Need free-form generated text or reasoning?* Reach for an LLM + LoRA.

## Step 5 — Feature design: turn a lead into text

DistilBERT reads **text**, not dictionaries. So we convert each lead's fields into one clear, consistent English-ish sentence. This is the **formatting step from notebook 10** — the model can only learn from what we put in the string, so we include every useful field in a fixed order.

We also define the **label map** that turns the words into the integers the model needs:
`{"cold": 0, "warm": 1, "hot": 2}` (and an inverse map to turn predictions back into words).

In [ ]:
LABEL2ID = {"cold": 0, "warm": 1, "hot": 2}   # words -> numbers (model needs numbers)
ID2LABEL = {v: k for k, v in LABEL2ID.items()} # numbers -> words (for readable output)

def lead_to_text(lead):
    """Turn one lead dict into a single consistent text string for the model."""
    return (
        f"family size {lead['family_size']}, "
        f"income {lead['income']}, "
        f"{lead['rent_or_own']} home, "
        f"CTA {lead['cta']}, "
        f"month {lead['engagement_month']}, "
        f"condition {lead['current_condition']}"
    )

# Show it on the first training lead:
example = train_leads[0]
print("Fields:", example)
print("\nAs text:", lead_to_text(example))
print("Label   :", example["lead_intent"], "-> id", LABEL2ID[example["lead_intent"]])

**What this does:** every lead becomes a line like
`family size 4, income 85000, own home, CTA requested_quote, month May, condition urgent`.
Consistent wording and order matter — the model learns associations like *"requested_quote" + "urgent" → hot* from the repeated structure.

**For comparison — the LLM (Path B) prompt template from notebook 10.** If you went the generative-LLM route instead, you'd format each example as an *instruction* with a target answer, like this (shown for understanding, not run here):

```text
### Instruction:
Classify this sales lead's intent as exactly one of: cold, warm, hot.

### Lead:
family size 4, income 85000, own home, CTA requested_quote, month May, condition urgent

### Response:
hot
```

Notice the difference: the classifier just needs the *features text* + an integer label, while the LLM needs the *whole instruction wrapper* and a *text* answer. Same data, different packaging.

## Step 6 — Tokenize and build the datasets

Models don't read letters; they read **token IDs** (notebook 07). The tokenizer that ships with DistilBERT chops our text into sub-word tokens and maps them to numbers. We:

1. Load `AutoTokenizer` for `distilbert-base-uncased`.
2. Build a Hugging Face `datasets.Dataset` for train and val (each row = `{"text": ..., "label": ...}`).
3. Run `.map()` to tokenize every row.

**About `max_length`.** Our lead strings are short and *fixed in shape* — they never exceed a couple dozen tokens. So we cap at `max_length=64`: long enough to never cut off a lead, short enough that training stays fast (padding everything to 64 wastes no real time). Picking the smallest length that fits your data is a free speed-up.

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Loaded tokenizer for", MODEL_NAME)

def to_records(rows):
    """Convert raw lead dicts into {text, label} records the model expects."""
    return [{"text": lead_to_text(r), "label": LABEL2ID[r["lead_intent"]]} for r in rows]

train_ds = Dataset.from_list(to_records(train_leads))
val_ds   = Dataset.from_list(to_records(val_leads))
print(train_ds)
print("Example record:", train_ds[0])

In [ ]:
MAX_LEN = 64   # plenty for our short lead strings; keeps training fast

def tokenize_batch(batch):
    """Tokenize a batch of texts: pad/truncate to MAX_LEN."""
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=MAX_LEN)

# .map(batched=True) tokenizes many rows at once (faster than one-by-one).
train_tok = train_ds.map(tokenize_batch, batched=True)
val_tok   = val_ds.map(tokenize_batch,   batched=True)

# Tell the dataset which columns are model inputs and to hand back torch tensors.
cols = ["input_ids", "attention_mask", "label"]
train_tok.set_format(type="torch", columns=cols)
val_tok.set_format(type="torch", columns=cols)

print("Tokenized. One row's keys:", list(train_tok[0].keys()))
print("input_ids length (should equal MAX_LEN):", len(train_tok[0]["input_ids"]))

**What this does:** turns each lead string into a fixed-length list of `input_ids` plus an `attention_mask` (which tells the model which positions are real vs padding). After `set_format("torch")`, the `Trainer` can feed these straight into the model. The `label` column rides along untouched — that's the answer the model is graded on.

## Step 7 — Fine-tune the model

Now the main event. We load `AutoModelForSequenceClassification` with **`num_labels=3`** — this takes pretrained DistilBERT and adds a fresh 3-way classification head on top. Fine-tuning teaches the whole thing to map our lead text to the right bucket.

> **Full fine-tuning vs LoRA.** DistilBERT is small (~66M params), so we just fine-tune it fully — simple and fast on CPU. For a *big* generative LLM you'd instead use **LoRA** (notebook 11) to train only a few tiny adapter layers and save huge amounts of memory. Same goal, cheaper method. At the end of this step we show the one-liner that would wrap this model with LoRA, so you can see how it'd plug in.

We pass our settings through `TrainingArguments`. **Every hyperparameter explained below** — read this, it's the heart of fine-tuning.

### The hyperparameters, in plain English

| Setting | What it controls | Our value & why |
|---|---|---|
| `learning_rate` | how big a step the model takes each update. Too high → it overshoots and never settles; too low → it crawls. | `2e-5` — the classic safe default for fine-tuning BERT-family models. |
| `num_train_epochs` | how many times the model sees the *entire* training set. More = more learning, but too many = **overfitting** (memorizing instead of generalizing). | `4` — enough to learn our pattern from 160 examples without badly overfitting. |
| `per_device_train_batch_size` | how many examples are processed together before one update. Bigger = smoother but more memory. | `8` — small and CPU-friendly. |
| `gradient_accumulation_steps` | a trick to *simulate* a bigger batch by summing several small batches before updating — useful when memory is tight. | `1` — our batches are tiny already, so no accumulation needed. |
| `MAX_LEN` (set earlier) | max tokens per example. Shorter = faster. | `64` — fits our short leads. |
| `warmup_ratio` | the fraction of training where the learning rate **ramps up from 0** before settling, so early updates don't shock the fresh head. | `0.1` (10% of steps). |
| `weight_decay` | gently shrinks weights each step to discourage overfitting (a mild regularizer). | `0.01` — standard. |

> **Expected runtime:** on a typical laptop **CPU**, this trains in roughly **1–4 minutes**. On a GPU it's seconds. The first run also downloads DistilBERT (~250 MB) once.

In [ ]:
from transformers import (AutoModelForSequenceClassification,
                          TrainingArguments, Trainer)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,                 # cold / warm / hot
    id2label=ID2LABEL,            # so the saved model knows the label names
    label2id=LABEL2ID,
)

# --- OPTIONAL LoRA path (commented out): how you'd shrink training for a big model ---
# from peft import LoraConfig, get_peft_model, TaskType
# lora_cfg = LoraConfig(task_type=TaskType.SEQ_CLS, r=8, lora_alpha=16,
#                       lora_dropout=0.05, target_modules=["q_lin", "v_lin"])
# model = get_peft_model(model, lora_cfg)
# model.print_trainable_parameters()   # would show only a tiny % of params train
# (We keep full fine-tuning here because DistilBERT is already small. See nb 11.)

print("Model loaded with a fresh 3-class head.")

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    """Trainer calls this after each eval: turn raw logits into accuracy + F1."""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)               # pick the highest-scoring class
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),  # treats all classes equally
    }

args = TrainingArguments(
    output_dir="lead_intent_model",          # where checkpoints/logs go
    learning_rate=2e-5,
    num_train_epochs=4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=1,
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy="epoch",                   # evaluate on val set after every epoch
    save_strategy="no",                      # skip saving checkpoints (we save manually at the end)
    report_to="none",                        # no external logging services
    seed=42,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    compute_metrics=compute_metrics,
)
print("Trainer ready. Starting training (1-4 min on CPU)...")
trainer.train()

**What this does:** `trainer.train()` runs the real fine-tuning loop you wrote by hand back in notebook 05 — forward pass, compute loss, backprop, update — now automated. Watch the printed table: the **training loss should fall** and the **validation accuracy/F1 should rise** across the 4 epochs. If validation stops improving while training loss keeps dropping, that's the early sign of overfitting.

> Don't be alarmed if accuracy isn't 99% — remember we only have 160 training leads. We'll measure exactly how good it is next.

## Step 8 — Evaluate properly

Accuracy alone can lie, especially with imbalanced classes (notebook 13). So we compute the **full picture** on the held-out validation set:

- **precision / recall / F1 per class** via `classification_report`,
- a **confusion matrix** plot (where do mistakes go?),
- a **manual-review table** of individual predictions, and
- a comparison against a **majority-class baseline** (always guess the most common label) — our model must beat this to be worth anything.

First we get the model's predictions on the validation set.

In [ ]:
pred_output = trainer.predict(val_tok)        # run the trained model on the val set
val_logits = pred_output.predictions
y_true = pred_output.label_ids                # the real labels
y_pred = np.argmax(val_logits, axis=-1)       # the model's guesses

label_names = [ID2LABEL[i] for i in range(3)] # ["cold","warm","hot"]
print("Predicted ids:", y_pred[:15])
print("True ids:     ", y_true[:15])

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

print("=== Classification report (validation set) ===")
print(classification_report(y_true, y_pred, target_names=label_names, digits=3, zero_division=0))

**What this does:** for each class it reports **precision** (of the leads we *called* hot, how many really were), **recall** (of the *actually* hot leads, how many we caught), and **F1** (their balance). The `macro avg` row treats all three classes as equally important — the fairest single number when classes are imbalanced.

In [ ]:
# --- Confusion matrix plot: rows = true label, columns = predicted label ---
cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(3)); ax.set_xticklabels(label_names)
ax.set_yticks(range(3)); ax.set_yticklabels(label_names)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title("Confusion matrix (validation set)")
# write the count inside each cell
for i in range(3):
    for j in range(3):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

**What this does:** the diagonal cells (top-left to bottom-right) are **correct** predictions; everything off-diagonal is a mistake. Look at *where* the errors land — confusing `warm` with `hot` is a near-miss, but confusing `cold` with `hot` would be a costly blunder for a sales team. A good model keeps mistakes close to the diagonal.

In [ ]:
# --- Manual-review table: eyeball a few individual predictions ---
review = pd.DataFrame({
    "lead_text": [r["text"] for r in to_records(val_leads)],
    "true":      [ID2LABEL[t] for t in y_true],
    "predicted": [ID2LABEL[p] for p in y_pred],
})
review["correct"] = review["true"] == review["predicted"]
print("First 12 validation predictions:")
display(review.head(12))

print("\nA few of the model's MISTAKES (if any):")
display(review[~review["correct"]].head(6))

**What this does:** numbers tell you *how much* is wrong; this table tells you *what kind* is wrong. Read a few mistakes and ask whether *you* could have classified them confidently from the text alone — often the "wrong" ones are genuinely borderline leads (a `warm` that's almost `hot`), which is reassuring. Clear, repeated errors on obvious leads would instead point to a data or formatting bug.

In [ ]:
# --- Baseline: always predict the most common class in the TRAINING data ---
from collections import Counter
majority_label_id = Counter(int(r["label"]) for r in to_records(train_leads)).most_common(1)[0][0]
baseline_pred = np.full_like(y_true, majority_label_id)

baseline_acc = accuracy_score(y_true, baseline_pred)
model_acc    = accuracy_score(y_true, y_pred)
print(f"Majority-class baseline accuracy: {baseline_acc:.3f}  (always guesses '{ID2LABEL[majority_label_id]}')")
print(f"Our fine-tuned model accuracy:    {model_acc:.3f}")
print(f"Improvement over baseline:        {model_acc - baseline_acc:+.3f}")

**What this does:** the baseline is the laziest possible "model." If our fine-tuned model can't beat *always guessing the most common class*, it has learned nothing useful. Beating the baseline by a clear margin is the real proof that fine-tuning paid off.

### ✏️ Exercise — squeeze out more accuracy
Try one change and re-run Steps 7–8: bump `num_train_epochs` to `6`, or raise `learning_rate` to `3e-5`, or generate `400` leads instead of `200` back in Step 2. Which helps most? (Spoiler from real life: **more data** usually wins.)

## Step 9 — Inference on brand-new leads

A model is only useful if you can *call it on new data* (notebook 14). We wrap the whole pipeline — format → tokenize → forward pass → softmax → label — into one friendly `predict(lead_dict)` function and run it on a few made-up leads the model has **never seen**.

In [ ]:
import torch.nn.functional as F

model.eval()                                  # turn off training-only behavior (dropout, etc.)

def predict(lead_dict):
    """Classify one lead. Returns (predicted_label, {label: probability})."""
    text = lead_to_text(lead_dict)            # Step 5 formatting
    enc = tokenizer(text, truncation=True, padding="max_length",
                    max_length=MAX_LEN, return_tensors="pt")
    with torch.no_grad():                     # no gradients needed for prediction = faster
        logits = model(**enc).logits          # raw scores
    probs = F.softmax(logits, dim=-1)[0]      # turn scores into probabilities that sum to 1
    pred_id = int(torch.argmax(probs))        # highest-probability class
    prob_map = {ID2LABEL[i]: round(float(probs[i]), 3) for i in range(3)}
    return ID2LABEL[pred_id], prob_map

# Three brand-new, hand-written leads:
new_leads = [
    {"family_size": 5, "income": 150000, "rent_or_own": "own",
     "cta": "requested_quote", "engagement_month": "Jun", "current_condition": "urgent"},   # looks HOT
    {"family_size": 1, "income": 25000, "rent_or_own": "rent",
     "cta": "newsletter_signup", "engagement_month": "Feb", "current_condition": "just_browsing"},  # looks COLD
    {"family_size": 3, "income": 70000, "rent_or_own": "own",
     "cta": "downloaded_brochure", "engagement_month": "Sep", "current_condition": "exploring"},  # WARM-ish
    {"family_size": 4, "income": 85000, "rent_or_own": "rent",
     "cta": "booked_demo", "engagement_month": "May", "current_condition": "exploring"},     # WARM/HOT borderline
]

for lead in new_leads:
    label, probs = predict(lead)
    print(f"\nLead: {lead_to_text(lead)}")
    print(f"  -> predicted intent: {label.upper()}   probabilities: {probs}")

**What this does:** for each new lead it prints the predicted bucket *and* the probability spread. The probabilities are gold for a sales team: a `hot` at 0.95 is a confident "call now," while a `hot` at 0.45 (with `warm` at 0.40) is a coin-flip the team might double-check. **Confidence, not just the label, is what makes predictions actionable.**

### ✏️ Exercise — try to fool it
Write your own `lead_dict` that you think is clearly `hot` or clearly `cold` and run `predict()` on it. Does the model agree with your intuition? When it disagrees, look back at the scoring rules in Step 2 — the model learned *those* patterns, not yours.

## Step 10 — Save the model (and how you'd serve it)

Finally, we **persist** the fine-tuned model and tokenizer with `save_pretrained` so you can reload them later without retraining (notebook 14). Saving both together is essential — the model and the tokenizer that produced its inputs must always travel as a pair.

In [ ]:
SAVE_DIR = "lead_intent_model_final"
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("Saved model + tokenizer to:", os.path.abspath(SAVE_DIR))
print("Files:", os.listdir(SAVE_DIR))

# Reload to prove it works (this is what a server would do at startup):
from transformers import AutoModelForSequenceClassification, AutoTokenizer
reloaded_model = AutoModelForSequenceClassification.from_pretrained(SAVE_DIR)
reloaded_tok   = AutoTokenizer.from_pretrained(SAVE_DIR)
print("\nReloaded successfully. id2label:", reloaded_model.config.id2label)

**What this does:** writes the weights, config, and tokenizer files to a folder and then loads them back to confirm the round-trip works. That folder is now a self-contained, shippable model.

**Serving it (from notebook 14).** To turn this into a live service, you'd load the saved folder inside a small **FastAPI** app and expose a `/predict` endpoint — roughly:

```python
# app.py  (sketch — see notebook 14 for the full version)
from fastapi import FastAPI
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch, torch.nn.functional as F

app = FastAPI()
model = AutoModelForSequenceClassification.from_pretrained("lead_intent_model_final")
tok   = AutoTokenizer.from_pretrained("lead_intent_model_final")
model.eval()

@app.post("/predict")
def predict_api(lead: dict):
    text = (f"family size {lead['family_size']}, income {lead['income']}, "
            f"{lead['rent_or_own']} home, CTA {lead['cta']}, "
            f"month {lead['engagement_month']}, condition {lead['current_condition']}")
    enc = tok(text, return_tensors="pt", truncation=True, max_length=64)
    with torch.no_grad():
        probs = F.softmax(model(**enc).logits, dim=-1)[0]
    pid = int(probs.argmax())
    return {"intent": model.config.id2label[pid], "confidence": float(probs[pid])}

# then run:  uvicorn app:app --reload
```

That's a deployable lead-scoring microservice — built from the model *you* just fine-tuned.

## Common mistakes & how to debug them

Things that trip people up on a project like this, and the fix:

- **Labels and IDs out of sync.** If your `LABEL2ID` / `id2label` don't match between training and inference, you'll get nonsense predictions that *look* valid. Always define the maps once and reuse them (we did).
- **Accuracy stuck at the baseline.** The model isn't learning. Try: more epochs, a slightly higher learning rate, more training data, or check your text-formatting actually includes the useful fields.
- **Validation accuracy way below training accuracy.** Classic **overfitting** — too many epochs for too little data. Reduce epochs, add `weight_decay`, or get more data.
- **`max_length` too short.** If you truncate away real signal, the model is blind to it. Make sure `MAX_LEN` covers your longest formatted example (ours is short, so 64 is safe).
- **Forgetting `model.eval()` / `torch.no_grad()` at inference.** Leaving training mode on makes predictions noisy and slower. Our `predict()` sets both.
- **Class imbalance fooling you.** A 70%-`cold` dataset gives 70% accuracy for free. Always compare to the majority-class baseline and read the **per-class F1**, not just accuracy.
- **Not saving the tokenizer.** Saving the model but not the tokenizer means you can't reproduce the inputs later. Always `save_pretrained` both.
- **Tiny dataset wobble.** With 200 examples, results jitter between runs. Don't over-interpret a 2% change — the seed and the small size both matter.

## Summary

You just built a complete fine-tuning project from nothing:

1. **Defined** a real business problem — rank sales leads by intent.
2. **Generated & inspected** 200 labelled leads and checked class balance.
3. **Split & saved** them as reproducible `leads_train.jsonl` / `leads_val.jsonl`.
4. **Chose** the right tool — a small DistilBERT classifier over a heavier LLM+LoRA, with reasons.
5. **Formatted** each lead into consistent text and mapped labels to integers.
6. **Tokenized** the data and built Hugging Face datasets.
7. **Fine-tuned** `AutoModelForSequenceClassification` with carefully explained hyperparameters — on CPU, in minutes.
8. **Evaluated** honestly: classification report, confusion matrix, manual review, and a baseline comparison.
9. **Ran inference** on brand-new leads with a reusable `predict()` that returns label + confidence.
10. **Saved** the model + tokenizer and sketched a FastAPI service to deploy it.

The result is a model that **beats the majority-class baseline** and produces actionable, confidence-scored predictions. And the headline lesson is honest: with only 200 examples it's good, not perfect — **the single biggest lever for a better model is more (and more realistic) data.**

This is the same pipeline shape — *prepare → format → fine-tune → evaluate → infer → deploy* — behind essentially every fine-tuning project, from a toy classifier to a production LLM. You now own all of it.

## What to learn next

**Congratulations — you finished the course and shipped a real fine-tuned model.** That's a genuine, employable skill. Here's where to take it next:

- **Use a bigger / better model.** Swap `distilbert-base-uncased` for `bert-base-uncased` or `roberta-base` and compare. More capacity can help — if you have the data to feed it.
- **Bring real data.** Replace the synthetic generator with an actual CSV export of your leads. Watch how messier, real-world data changes everything (the lesson from notebook 09 in full force).
- **Try QLoRA on a GPU.** Take the optional LLM path (Path B) and fine-tune a small generative model with **QLoRA** (notebooks 11–12) on a free Colab GPU. See how far you can push quality per dollar.
- **Deploy it for real.** Build out the FastAPI service from **notebook 14**, containerize it, and call it from a tiny web form. Now your model is a product.
- **Go deeper into alignment.** Read up on **RLHF** and **DPO** — the techniques that turn a raw fine-tuned model into a helpful, well-behaved assistant. They're the next frontier after supervised fine-tuning.
- **Further reading.** The official **Hugging Face Course** (huggingface.co/learn) is the best free next step — it expands on every tool you used here, with more datasets and tasks.

You started with basic Python. You now understand data prep, formatting, LoRA/QLoRA, the `Trainer`, evaluation, inference, and deployment — and you've used them together on a working project. Go fine-tune something you care about. 🚀